In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [ ]:
# Load pre-trained model for segmentation
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils
pose = mp_pose.Pose()


# Function to find the dominant color in an image region
def extract_dominant_color(image, k=4):
    """
    Extract the dominant color using KMeans clustering.

    Args:
        image (np.array): Cropped image part.
        k (int): Number of clusters.

    Returns:
        tuple: Dominant color in RGB format.
    """
    print(0)
    print(image.shape)
    plt.imshow(image)
    plt.show()
    pixels = image.reshape(-1, 3)
    print(pixels.shape)
    # print(1)
    # kmeans = KMeans(n_clusters=k)
    # print(2)
    # kmeans.fit(pixels) # <-- Here is the error
    # print(3)
    # dominant_color = kmeans.cluster_centers_[0]
    # print(4)
    # return dominant_color.astype(int)

    # Function to display the color


def show_color(color):
    """
    Display the extracted color.

    Args:
        color (tuple): RGB color.
    """
    plt.imshow([[color]])
    plt.axis("off")
    plt.show()


# Function to classify body physique based on the bounding box
def classify_body_physique(landmarks):
    """
    Classify body physique based on the shoulder, hip, and height ratios.

    Args:
        landmarks: Pose landmarks.

    Returns:
        str: Body physique classification.
    """
    shoulders_width = np.linalg.norm(
        np.array(
            [
                landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y,
            ]
        )
        - np.array(
            [
                landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,
                landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y,
            ]
        )
    )
    hips_width = np.linalg.norm(
        np.array(
            [
                landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y,
            ]
        )
        - np.array(
            [
                landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x,
                landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y,
            ]
        )
    )
    height = np.linalg.norm(
        np.array(
            [
                landmarks[mp_pose.PoseLandmark.NOSE.value].x,
                landmarks[mp_pose.PoseLandmark.NOSE.value].y,
            ]
        )
        - np.array(
            [
                landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x,
                landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y,
            ]
        )
    )

    ratio = shoulders_width / hips_width

    if ratio > 1.1:
        return "Inverted Triangle"
    elif ratio < 0.9:
        return "Pear"
    else:
        return "Rectangle"


# Load an image
image_path = "../tmp/person2.jpeg"  # Replace with your image path
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Detect pose landmarks
results = pose.process(image_rgb)
landmarks = results.pose_landmarks.landmark

# Draw pose landmarks
if results.pose_landmarks:
    mp_draw.draw_landmarks(image_rgb, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

    plt.imshow(image_rgb)
    plt.show()

    # get tshirt color
    left_shoulder_x = int(
        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x * image_rgb.shape[1]
    )
    left_shoulder_y = int(
        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y * image_rgb.shape[0]
    )
    right_shoulder_x = int(
        landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x * image_rgb.shape[1]
    )
    right_shoulder_y = int(
        landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y * image_rgb.shape[0]
    )

    left_hip_x = int(
        landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x * image_rgb.shape[1]
    )
    left_hip_y = int(
        landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y * image_rgb.shape[0]
    )
    right_hip_x = int(
        landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x * image_rgb.shape[1]
    )
    right_hip_y = int(
        landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y * image_rgb.shape[0]
    )

    tshirt_edges = [
        (right_shoulder_x, right_shoulder_y),
        (left_shoulder_x, left_shoulder_y),
        (right_hip_x, right_hip_y),
        (left_hip_x, left_hip_y),
    ]

    tshirt_edges.sort(key=lambda t: t[0] * t[1])

    tshirt = image_rgb[
        tshirt_edges[0][1] : tshirt_edges[-1][1],
        tshirt_edges[0][0] : tshirt_edges[-1][0],
    ]

    plt.imshow(tshirt)
    plt.show()

    # get pants
    left_knee_x = int(
        landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x * image_rgb.shape[1]
    )
    left_knee_y = int(
        landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y * image_rgb.shape[0]
    )

    pants_edges = [(left_hip_x, left_hip_y), (left_knee_x, left_knee_y)]

    pants_edges.sort(key=lambda t: t[0] * t[1])

    if pants_edges[0][0] == pants_edges[-1][0]:
        pants = image_rgb[
            pants_edges[0][1] : pants_edges[-1][1],
            pants_edges[0][0] : pants_edges[-1][0] + 1,
        ]
    else:
        pants = image_rgb[
            pants_edges[0][1] : pants_edges[-1][1],
            pants_edges[0][0] : pants_edges[-1][0],
        ]

    plt.imshow(pants)
    plt.show()

    # Extract key areas
    # landmarks = results.pose_landmarks.landmark
    # left_shoulder_x = int(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x*image_rgb.shape[1])
    # left_shoulder_y = int(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y*image_rgb.shape[0])
    # right_shoulder_x = int(landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x*image_rgb.shape[1])
    # right_shoulder_y = int(landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y*image_rgb.shape[0])
    # nose_x = int(landmarks[mp_pose.PoseLandmark.NOSE.value].x*image_rgb.shape[1])
    # nose_y = int(landmarks[mp_pose.PoseLandmark.NOSE.value].y*image_rgb.shape[0])

    # get hair location

    # print("Nose", nose_x, nose_y)
    # print("Left Shoulder", left_shoulder_x, left_shoulder_y)

    # # 0°
    # if right_shoulder_y > nose_y and right_shoulder_x < nose_x:
    #     pass

    # # 90°
    # elif right_shoulder_y < nose_y and right_shoulder_x < nose_x:
    #     image_rgb = np.rotate(image_rgb, -90)

    # # 180°
    # elif right_shoulder_y < nose_y and right_shoulder_x > nose_x:

    # # 270°
    # elif right_shoulder_y > nose_y and right_shoulder_x > nose_x:

    # shoulder_x_median = (right_shoulder_x + left_shoulder_x)//2
    # hair = image_rgb[int(nose_y*0.1):int(nose_y*0.50), shoulder_x_median-int(0.05*image_rgb.shape[1]):shoulder_x_median+int(0.05*image_rgb.shape[1])]
    # plt.imshow(hair)
    # plt.show()

    # head = image_rgb[int(nose_y*image_rgb.shape[0]-50):int(nose_y*image_rgb.shape[0]+50), int(nose_x*image_rgb.shape[1]-50):int(nose_x*image_rgb.shape[1]+50)]
    # plt.imshow(head)
    # plt.imshow(image_rgb)
    # plt.show()

    # # Extract colors of specific regions (hair, skin, shirt, pants)
    # hair = image_rgb[0:100, 0:100]
    # hair_color = extract_dominant_color(hair)  # Simplified; customize area based on landmarks

    # Classify body physique
    # body_type = classify_body_physique(landmarks)
    # print("Body Physique:", body_type)